In [3]:
pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.6 MB 7.4 MB/s eta 0:00:02
   ------------ --------------------------- 2.9/9.6 MB 7.3 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.6 MB 7.2 MB/s eta 0:00:01
   ------------------------- -------------- 6.0/9.6 MB 7.2 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 7.3 MB/s eta 0:00:01
   ------------------------------------- -- 8.9/9.6 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 7.1 MB/s  0:00:01

   ---------------------------------------- 0/2 [patsy]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- -----------

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

# ============================================================
# STEP 1: Load data
# ============================================================
df = pd.read_csv('annotations_indoor_outdoor.csv')
print(f'Images in sample: {len(df)}')
print(df.head())

rater_cols = ['annotator_MED', 'annotator_NB', 'annotator_LE']
df = df.dropna(subset=rater_cols)
print(f'Fully labelled images: {len(df)}')

# ============================================================
# STEP 2: Cohen's Kappa (pairwise)
# ============================================================
k_MED_NB = cohen_kappa_score(df['annotator_MED'], df['annotator_NB'])
k_MED_LE = cohen_kappa_score(df['annotator_MED'], df['annotator_LE'])
k_NB_LE  = cohen_kappa_score(df['annotator_NB'],  df['annotator_LE'])

print(f"Cohen's kappa (MED vs NB): {k_MED_NB:.3f}")
print(f"Cohen's kappa (MED vs LE): {k_MED_LE:.3f}")
print(f"Cohen's kappa (NB  vs LE): {k_NB_LE:.3f}")
print(f"Mean pairwise Cohen's kappa: {np.mean([k_MED_NB, k_MED_LE, k_NB_LE]):.3f}")

# Raw Agreement
agree_MED_NB = (df['annotator_MED'] == df['annotator_NB']).mean()
agree_MED_LE = (df['annotator_MED'] == df['annotator_LE']).mean()
agree_NB_LE  = (df['annotator_NB']  == df['annotator_LE']).mean()

print(f'Raw agreement (MED vs NB): {agree_MED_NB:.1%}')
print(f'Raw agreement (MED vs LE): {agree_MED_LE:.1%}')
print(f'Raw agreement (NB  vs LE): {agree_NB_LE:.1%}')

# Confusion Matrix (MED vs NB)
labels = sorted(df['annotator_MED'].unique())
cm = confusion_matrix(df['annotator_MED'], df['annotator_NB'], labels=labels)
print('\nConfusion Matrix (Rows=MED, Cols=NB):')
print(pd.DataFrame(cm, index=labels, columns=labels))

# ============================================================
# STEP 3: Fleiss' Kappa (all three annotators)
# ============================================================
rater_data = df[rater_cols].values
counts, categories = aggregate_raters(rater_data)
print(f'\nCategories detected: {categories}')

kappa_fleiss = fleiss_kappa(counts, method='fleiss')
print(f"Fleiss' kappa (3 annotators): {kappa_fleiss:.3f}")

# ============================================================
# STEP 4: Save disagreements
# ============================================================
def all_agree(row):
    return len(set([row['annotator_MED'], row['annotator_NB'], row['annotator_LE']])) == 1

df['agreement'] = df.apply(all_agree, axis=1)
disagreements = df[~df['agreement']]
print(f'\nImages with disagreement: {len(disagreements)} of {len(df)}')

disagreements.to_csv('disagreements_to_reconcile.csv', index=False)
print('✅ disagreements_to_reconcile.csv saved')

# ============================================================
# STEP 5: Save metrics summary
# ============================================================
metrics = {
    "Metric": [
        "Cohen's kappa (MED vs NB)",
        "Cohen's kappa (MED vs LE)",
        "Cohen's kappa (NB vs LE)",
        "Mean pairwise Cohen's kappa",
        "Raw agreement (MED vs NB)",
        "Raw agreement (MED vs LE)",
        "Raw agreement (NB vs LE)",
        "Fleiss' kappa (3 annotators)",
        "Images with disagreement",
        "Total images"
    ],
    "Value": [
        round(k_MED_NB, 3),
        round(k_MED_LE, 3),
        round(k_NB_LE, 3),
        round(np.mean([k_MED_NB, k_MED_LE, k_NB_LE]), 3),
        f'{agree_MED_NB:.1%}',
        f'{agree_MED_LE:.1%}',
        f'{agree_NB_LE:.1%}',
        round(kappa_fleiss, 3),
        len(disagreements),
        len(df)
    ]
}

pd.DataFrame(metrics).to_csv('agreement_metrics_indoor_outdoor.csv', index=False)
print('✅ agreement_metrics.csv saved')

Images in sample: 300
          filename annotator_NB annotator_LE gold_standard annotator_CLIP
0  image_10020.jpg       indoor       indoor        indoor        outdoor
1  image_10026.jpg      outdoor      outdoor       outdoor        outdoor
2   image_1009.jpg      outdoor      outdoor       outdoor        outdoor
3  image_10104.jpg      outdoor      outdoor       outdoor        outdoor
4  image_10134.jpg      outdoor      outdoor       outdoor        outdoor


KeyError: ['annotator_MED']

['filename', 'annotator_NB', 'annotator_LE']
